In [1]:
# ═══════════════════════════════════════════════════════════
# STEP 1 — Mount Google Drive & configure paths
# ═══════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Query Quality Evaluation & MMR Diversity Filtering

**Pipeline:**
1. Load `clean_data.csv` (book corpus) and `embedding_dataset.jsonl` (generated queries)
2. Embed book descriptions + queries with **BGE-M3**
3. Compute per-query quality metrics (relevance, length, uniqueness)
4. Apply **Maximal Marginal Relevance (MMR)** to select diverse queries per book
5. Save filtered dataset to `embedding_dataset_filtered.jsonl`
6. Summary statistics

In [2]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE  = '/content/drive/MyDrive'


# ── Paths ──────────────────────────────────────────────────────────────────
# BASE = Path("../data")
CSV_PATH    = f"{DRIVE_BASE}/clean_data.csv"
JSONL_PATH  = f"{DRIVE_BASE}/embedding_dataset.jsonl"
OUTPUT_PATH = f"{DRIVE_BASE}/embedding_dataset_filtered.jsonl"

# ── Parameters ─────────────────────────────────────────────────────────────
EMBEDDING_MODEL = "BAAI/bge-m3"   # multilingual; good for Vietnamese
TOP_K   = 3     # number of queries to keep per book after MMR
LAMBDA  = 0.7   # MMR trade-off: 1.0 = pure relevance, 0.0 = pure diversity

# Quality thresholds
MIN_QUERY_LEN  = 10   # characters
MAX_QUERY_LEN  = 200
MIN_RELEVANCE  = 0.4  # cosine similarity to book description

print("Config loaded.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Config loaded.


## 1 · Load data

In [3]:
books_df = pd.read_csv(CSV_PATH)
books_df = books_df[:5000]
books_df['chunk'] = (
    "Title: " + books_df["Title"] + " | " +
    "Authors: " + books_df["Authors"] + " | " +
    "Description: " + books_df["Description"] + " | " +
    "Category: " + books_df["Category"]
)# books_df.to_csv(CSV_PATH, index=False)

with open(JSONL_PATH, "r", encoding="utf-8") as f:
    raw_records = [json.loads(line) for line in f]

# Flatten to a DataFrame: one row per query
rows = []
for rec in raw_records:
    for q in rec["queries"]:
        rows.append({"book_id": rec["book_id"], "query": q})
queries_df = pd.DataFrame(rows)

print(f"Books  : {len(books_df):,}")
print(f"Records: {len(raw_records)} books with queries")
print(f"Queries: {len(queries_df)} total  ({len(queries_df)/len(raw_records):.1f} per book avg)")
queries_df.head()

Books  : 5,000
Records: 4944 books with queries
Queries: 24715 total  (5.0 per book avg)


,book_id,query
0,0,sách thơ về cảm xúc tuổi thơ và niềm tin
1,0,có sách thơ nào viết về bệnh tật và hy vọng không
2,0,thơ viết về cái chết của anh em và thiên nhiên
3,0,gợi ý sách truyền cảm hứng của trẻ em khuyết tật
4,0,những vần thơ phản ánh cuộc sống và đức tin


## 2 · Embed with BGE-M3

In [4]:
import hashlib, torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# ── Cache paths (saved alongside the data on Drive) ────────────────────────
CACHE_BOOK_EMB   = f"{DRIVE_BASE}/cache_book_embs.npy"
CACHE_QUERY_EMB  = f"{DRIVE_BASE}/cache_query_embs.npy"
CACHE_META       = f"{DRIVE_BASE}/cache_meta.json"

def _cache_key() -> str:
    """Fingerprint that invalidates cache when model or data changes."""
    n_books   = len(books_df)
    n_queries = len(queries_df)
    raw = f"{EMBEDDING_MODEL}|books={n_books}|queries={n_queries}"
    return hashlib.md5(raw.encode()).hexdigest()

def _load_cache():
    """Return (book_embs, query_embs) from disk, or None if invalid/missing."""
    try:
        with open(CACHE_META, "r") as f:
            meta = json.load(f)
        if meta.get("key") != _cache_key():
            print("⚠️  Cache key mismatch — will re-embed.")
            return None
        book_embs  = np.load(CACHE_BOOK_EMB)
        query_embs = np.load(CACHE_QUERY_EMB)
        print(f"✅ Loaded embeddings from cache  "
              f"(books: {book_embs.shape}, queries: {query_embs.shape})")
        return book_embs, query_embs
    except FileNotFoundError:
        return None

def _save_cache(book_embs: np.ndarray, query_embs: np.ndarray):
    """Persist embeddings and metadata to Drive."""
    np.save(CACHE_BOOK_EMB,  book_embs)
    np.save(CACHE_QUERY_EMB, query_embs)
    with open(CACHE_META, "w") as f:
        json.dump({"key": _cache_key(), "model": EMBEDDING_MODEL,
                   "n_books": len(books_df), "n_queries": len(queries_df)}, f)
    print(f"💾 Embeddings cached → {DRIVE_BASE}/cache_*.npy")

# ── Embed or load from cache ────────────────────────────────────────────────
cached = _load_cache()

if cached is not None:
    book_embs, query_embs = cached
else:
    model = SentenceTransformer(EMBEDDING_MODEL, device=device)

    print(f"\nEmbedding {len(books_df)} book descriptions...")
    book_embs = model.encode(
        books_df["chunk"].tolist(),
        batch_size=32,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    print(f"\nEmbedding {len(queries_df)} queries...")
    query_embs = model.encode(
        queries_df["query"].tolist(),
        batch_size=64,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    _save_cache(book_embs, query_embs)

book_emb_map = {book_id: emb for book_id, emb in zip(books_df["book_id"], book_embs)}
queries_df["emb_idx"] = range(len(queries_df))
print("\nEmbedding ready.")

Using device: cpu
✅ Loaded embeddings from cache  (books: (5000, 1024), queries: (24715, 1024))

Embedding ready.


## 3 · Quality metrics

In [5]:
def compute_relevance(row):
    """Cosine similarity between the query embedding and its book description embedding."""
    book_emb = book_emb_map.get(row["book_id"])
    if book_emb is None:
        return np.nan
    q_emb = query_embs[row["emb_idx"]]
    return float(np.dot(q_emb, book_emb))  # both normalized → dot == cosine


queries_df["char_len"]    = queries_df["query"].str.len()
queries_df["word_count"]  = queries_df["query"].str.split().str.len()
queries_df["relevance"]   = queries_df.apply(compute_relevance, axis=1)

# Hard quality flags
queries_df["ok_length"]   = queries_df["char_len"].between(MIN_QUERY_LEN, MAX_QUERY_LEN)
queries_df["ok_relevance"] = queries_df["relevance"] >= MIN_RELEVANCE
queries_df["quality_ok"]  = queries_df["ok_length"] & queries_df["ok_relevance"]

print("=== Quality Metrics Summary ===")
print(f"  Total queries       : {len(queries_df)}")
print(f"  Fail length check   : {(~queries_df['ok_length']).sum()}")
print(f"  Fail relevance check: {(~queries_df['ok_relevance']).sum()}")
print(f"  Pass all checks     : {queries_df['quality_ok'].sum()} ({queries_df['quality_ok'].mean()*100:.1f}%)")
print()
print(queries_df[["char_len","word_count","relevance"]].describe().round(3))
queries_df.columns

=== Quality Metrics Summary ===
  Total queries       : 24715
  Fail length check   : 4
  Fail relevance check: 91
  Pass all checks     : 24622 (99.6%)

        char_len  word_count  relevance
count  24715.000   24715.000  24715.000
mean      59.741      13.548      0.571
std       12.937       2.593      0.062
min       16.000       2.000      0.263
25%       52.000      12.000      0.528
50%       59.000      13.000      0.571
75%       67.000      15.000      0.614
max      532.000      51.000      0.787


Index(['book_id', 'query', 'emb_idx', 'char_len', 'word_count', 'relevance',
       'ok_length', 'ok_relevance', 'quality_ok'],
      dtype='object')

In [6]:
# Check Diversity of queries
diversity_rows = []
for book_id, grp in queries_df.groupby("book_id"):
    emb_idx = grp["emb_idx"].tolist()
    mat  = query_embs[emb_idx]                     # (n_queries, dim)
    sim  = cosine_similarity(mat)               # (n, n)
    n    = len(emb_idx)
    # Average of upper triangle (excluding diagonal)
    avg_sim = (sim.sum() - n) / (n * (n - 1)) if n > 1 else 0.0
    diversity_rows.append({"book_id": book_id, "avg_pairwise_sim": avg_sim})

diversity_df = pd.DataFrame(diversity_rows)
print("=== Within-Book Query Diversity (before MMR) ===")
print(f"  Avg pairwise similarity : {diversity_df['avg_pairwise_sim'].mean():.4f}")
print(f"  (lower = more diverse)")
print()
print("Most redundant books (high similarity between queries):")
print(diversity_df.nlargest(5, "avg_pairwise_sim").merge(books_df[["book_id","Title"]], on="book_id")[["book_id","Title","avg_pairwise_sim"]].to_string(index=False))
print()
print("Most diverse books (low similarity between queries):")
print(diversity_df.nsmallest(5, "avg_pairwise_sim").merge(books_df[["book_id","Title"]], on="book_id")[["book_id","Title","avg_pairwise_sim"]].to_string(index=False))

=== Within-Book Query Diversity (before MMR) ===
  Avg pairwise similarity : 0.6791
  (lower = more diverse)

Most redundant books (high similarity between queries):
 book_id                                                                                                                       Title  avg_pairwise_sim
     597 Plasterworks: A Beginner's Guide to Molding and Decorating Plaster Projects from Stars and Cherubs to Shells and Sunflowers          0.892278
     552                                                                                      Holiday Cooking (Great Taste, Low Fat)          0.891113
    2686                                                                                         Needlework: Blocking and Finishing.          0.890858
    1875                                                                                                      Monkey Learns To Potty          0.886511
    1915                                                                       

## 4 · MMR selection

For each book, pick the `TOP_K` queries that maximize:

$$\text{MMR}(q) = \lambda \cdot \underbrace{\text{sim}(q, \text{book})}_\text{relevance} - (1-\lambda) \cdot \underbrace{\max_{s \in S} \text{sim}(q, s)}_\text{redundancy}$$

Queries that fail the hard quality checks are excluded before MMR.

In [7]:
def mmr_select(
    candidates: pd.DataFrame,
    query_embs: np.ndarray,
    book_emb: np.ndarray,
    top_k: int,
    lam: float,
) -> list[int]:
    """
    Greedy MMR selection.

    Args:
        candidates : DataFrame rows for this book (must have 'emb_idx').
        query_embs : All query embeddings array (normalized).
        book_emb   : Book description embedding (normalized).
        top_k      : Number of queries to select.
        lam        : Trade-off weight (higher = more relevance-focused).

    Returns:
        List of selected emb_idx values (in selection order).
    """
    emb_idx = candidates["emb_idx"].tolist()
    if not emb_idx:
        return []

    # Relevance scores: dot product with book embedding (both normalized)
    relevance = {i: float(np.dot(query_embs[i], book_emb)) for i in emb_idx}

    selected   = []
    remaining  = list(emb_idx)

    while remaining and len(selected) < top_k:
        if not selected:
            # First pick: highest relevance
            best = max(remaining, key=lambda i: relevance[i])
        else:
            selected_embs = query_embs[selected]  # (k, dim)
            best_score = -np.inf
            best = None
            for i in remaining:
                q_emb = query_embs[i]
                # Similarity to already-selected queries
                sims_to_selected = cosine_similarity(q_emb.reshape(1, -1), selected_embs)[0]
                max_sim = float(sims_to_selected.max())
                mmr_score = lam * relevance[i] - (1 - lam) * max_sim
                if mmr_score > best_score:
                    best_score = mmr_score
                    best = i
        selected.append(best)
        remaining.remove(best)

    return selected


# Run MMR per book
selected_records = []
rejected_records = []

for book_id, grp in queries_df.groupby("book_id"):
    book_emb = book_emb_map.get(book_id)
    if book_emb is None:
        continue

    # Filter to quality-passing candidates first
    candidates = grp[grp["quality_ok"]].copy()

    # Fallback: if too few pass quality, include all (sorted by relevance)
    if len(candidates) < TOP_K:
        candidates = grp.copy()

    chosen_emb_idx = mmr_select(candidates, query_embs, book_emb, TOP_K, LAMBDA)
    chosen_set  = set(chosen_emb_idx)

    chosen_queries = candidates[candidates["emb_idx"].isin(chosen_set)]["query"].tolist()
    selected_records.append({"book_id": book_id, "queries": chosen_queries})

    # Track rejected for analysis
    rejected = grp[~grp["emb_idx"].isin(chosen_set)]
    for _, row in rejected.iterrows():
        rejected_records.append({
            "book_id": book_id,
            "query": row["query"],
            "relevance": row["relevance"],
            "quality_ok": row["quality_ok"],
        })

selected_df = pd.DataFrame([
    {"book_id": r["book_id"], "query": q}
    for r in selected_records for q in r["queries"]
])
rejected_df = pd.DataFrame(rejected_records)

print(f"Selected : {len(selected_df)} queries ({len(selected_records)} books × {TOP_K})")
print(f"Rejected : {len(rejected_df)} queries")

Selected : 14829 queries (4943 books × 3)
Rejected : 9886 queries


## 5 · Post-MMR diversity comparison

In [8]:
# Re-compute within-book avg pairwise similarity on selected queries
diversity_after = []
for rec in selected_records:
    book_id = rec["book_id"]
    grp = selected_df[selected_df["book_id"] == book_id]
    emb_idx = queries_df[queries_df["query"].isin(grp["query"]) &
                      (queries_df["book_id"] == book_id)]["emb_idx"].tolist()
    if len(emb_idx) < 2:
        diversity_after.append({"book_id": book_id, "avg_pairwise_sim_after": np.nan})
        continue
    mat  = query_embs[emb_idx]
    sim  = cosine_similarity(mat)
    n    = len(emb_idx)
    avg  = (sim.sum() - n) / (n * (n - 1))
    diversity_after.append({"book_id": book_id, "avg_pairwise_sim_after": avg})

div_after_df = pd.DataFrame(diversity_after)

before_mean = diversity_df["avg_pairwise_sim"].mean()
after_mean  = div_after_df["avg_pairwise_sim_after"].mean()

print("=== Diversity Comparison ===")
print(f"  Avg pairwise sim BEFORE MMR : {before_mean:.4f}")
print(f"  Avg pairwise sim AFTER  MMR : {after_mean:.4f}")
print(f"  Improvement (↓ better)      : {before_mean - after_mean:+.4f}")

# Relevance before / after
before_rel = queries_df["relevance"].mean()
selected_emb_idx = queries_df[
    queries_df.apply(lambda r: r["query"] in set(selected_df[selected_df["book_id"]==r["book_id"]]["query"]), axis=1)
]["relevance"].mean()
# Simpler: join
merged = selected_df.merge(queries_df[["book_id","query","relevance"]], on=["book_id","query"])
after_rel = merged["relevance"].mean()
print()
print(f"  Avg relevance BEFORE MMR    : {before_rel:.4f}")
print(f"  Avg relevance AFTER  MMR    : {after_rel:.4f}")

=== Diversity Comparison ===
  Avg pairwise sim BEFORE MMR : 0.6791
  Avg pairwise sim AFTER  MMR : 0.6606
  Improvement (↓ better)      : +0.0185

  Avg relevance BEFORE MMR    : 0.5708
  Avg relevance AFTER  MMR    : 0.5858


## 6 · Inspect low-quality and rejected queries

In [9]:
# Books where all 5 queries failed quality check
fail_books = queries_df.groupby("book_id")["quality_ok"].sum()
all_fail = fail_books[fail_books == 0].index.tolist()
print(f"Books where NO query passed quality: {len(all_fail)}")
if all_fail:
    print(books_df[books_df["book_id"].isin(all_fail)][["book_id","Title","Category"]].to_string(index=False))

print()
print("=== Low-relevance queries (relevance < 0.3) ===")
low_rel = queries_df[queries_df["relevance"] < 0.3].merge( books_df[["book_id","Title"]], on="book_id" )[["book_id","Title","query","relevance"]].sort_values("relevance")
print(f"  Count: {len(low_rel)}")
print(low_rel.head(10).to_string(index=False))

Books where NO query passed quality: 0

=== Low-relevance queries (relevance < 0.3) ===
  Count: 3
 book_id                                                           Title                                                           query  relevance
    1670 Energy Breakthrough: Jump-start Your Weight Loss and Feel Great        thơ viết về cái chết của anh chị em và niềm tin tôn giáo   0.263403
     287                     Out to Canaan (Book 4 of the Mitford Years) cuộc đối đầu giữa kế hoạch phát triển đô thị và bảo tồn văn hóa   0.277825
    1714                                          America's Best Recipes        thơ viết về cái chết của anh chị em và niềm tin tôn giáo   0.283723


In [10]:
# Show example: before vs after MMR for a high-redundancy book
most_redundant_id = diversity_df.loc[diversity_df["avg_pairwise_sim"].idxmax(), "book_id"]
book_title = books_df[books_df["book_id"] == most_redundant_id]["Title"].values[0]

print(f"Most redundant book (id={most_redundant_id}): {book_title}")
print()
print("BEFORE MMR (all 5 queries):")
for q in queries_df[queries_df["book_id"] == most_redundant_id]["query"]:
    print(f"  • {q}")

print()
print(f"AFTER MMR (top {TOP_K}, λ={LAMBDA}):")
for rec in selected_records:
    if rec["book_id"] == most_redundant_id:
        for q in rec["queries"]:
            print(f"  ✓ {q}")

Most redundant book (id=597): Plasterworks: A Beginner's Guide to Molding and Decorating Plaster Projects from Stars and Cherubs to Shells and Sunflowers

BEFORE MMR (all 5 queries):
  • sách hướng dẫn làm nghệ thuật trang trí bằng vữa
  • có sách hướng dẫn làm đồ trang trí bằng vữa không
  • sách hướng dẫn làm đồ trang trí bằng vữa với dụng cụ thông thường
  • gợi ý sách hướng dẫn làm đồ trang trí bằng vữa cho người mới bắt đầu
  • sách hướng dẫn làm đồ trang trí bằng vữa với nhiều dự án từng bước

AFTER MMR (top 3, λ=0.7):
  ✓ sách hướng dẫn làm đồ trang trí bằng vữa với dụng cụ thông thường
  ✓ gợi ý sách hướng dẫn làm đồ trang trí bằng vữa cho người mới bắt đầu
  ✓ sách hướng dẫn làm đồ trang trí bằng vữa với nhiều dự án từng bước


## 7 · Save filtered dataset

In [11]:
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for rec in selected_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"✅ Saved {len(selected_records)} records → {OUTPUT_PATH}")
print(f"   {len(selected_df)} queries total  ({TOP_K} per book)")

✅ Saved 4943 records → /content/drive/MyDrive/embedding_dataset_filtered.jsonl
   14829 queries total  (3 per book)


## 8 · Summary report

In [12]:
print("━" * 55)
print("QUERY QUALITY & MMR REPORT")
print("━" * 55)
print(f"  Input  : {len(raw_records)} books, {len(queries_df)} queries")
print(f"  Output : {len(selected_records)} books, {len(selected_df)} queries")
print()
print("Quality (before MMR)")
print(f"  Pass rate            : {queries_df['quality_ok'].mean()*100:.1f}%")
print(f"  Avg relevance        : {before_rel:.4f}")
print(f"  Avg char length      : {queries_df['char_len'].mean():.1f}")
print(f"  Avg word count       : {queries_df['word_count'].mean():.1f}")
print()
print("Diversity (avg pairwise cosine similarity, ↓ better)")
print(f"  Before MMR           : {before_mean:.4f}")
print(f"  After  MMR           : {after_mean:.4f}  ({(before_mean-after_mean)/before_mean*100:+.1f}%)")
print()
print("Relevance (avg cosine sim to book description, ↑ better)")
print(f"  Before MMR           : {before_rel:.4f}")
print(f"  After  MMR           : {after_rel:.4f}  ({(after_rel-before_rel)/before_rel*100:+.1f}%)")
print("━" * 55)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
QUERY QUALITY & MMR REPORT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Input  : 4944 books, 24715 queries
  Output : 4943 books, 14829 queries

Quality (before MMR)
  Pass rate            : 99.6%
  Avg relevance        : 0.5708
  Avg char length      : 59.7
  Avg word count       : 13.5

Diversity (avg pairwise cosine similarity, ↓ better)
  Before MMR           : 0.6791
  After  MMR           : 0.6606  (+2.7%)

Relevance (avg cosine sim to book description, ↑ better)
  Before MMR           : 0.5708
  After  MMR           : 0.5858  (+2.6%)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


# 9 · Hard Negative Mining

Pipeline xây dựng training dataset cho BGE Embedding + Reranker:

1. Build **FAISS HNSW index** từ toàn bộ 5000 book embeddings (indexed ANN, local, không cần network)
2. Với mỗi query đã qua MMR filter: retrieve **top-50** candidates
3. Phân loại negatives theo rank: **Hard** (1–10) / **Medium** (11–30) / **Easy** (31+)
4. Bổ sung **same-category negatives** (khó vì cùng chủ đề)
5. Lọc **false negatives** (sách có cosine sim với positive > 0.85)
6. Chọn final **7 negatives/query** (4 hard + 2 same-cat + 1 medium)
7. Export BGE format `{query, pos, neg}` + split **train/val/test** theo `book_id`

In [13]:
! pip install -q faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 66.8 MB/s eta 0:00:00:00:0100:01


In [14]:
import os
import random

import faiss
from sklearn.model_selection import train_test_split

print(f"FAISS version: {faiss.__version__}")

# ── Hard negative mining parameters ────────────────────────────────────────
TOP_K_RETRIEVE      = 50
N_NEGATIVES         = 7
N_HARD              = 4    # rank 1–10
N_SAME_CAT          = 2    # cùng Category với positive
N_MEDIUM            = 1    # rank 11–30
HARD_RANK_MAX       = 10
MEDIUM_RANK_MAX     = 30
FALSE_NEG_THRESHOLD = 0.85 # cosine sim với positive > ngưỡng này → loại

# ── FAISS index parameters (HNSW — indexed ANN cho tốc độ truy vấn cao) ────
FAISS_HNSW_M           = 32    # số neighbor mỗi node trong HNSW graph
FAISS_EF_CONSTRUCTION  = 200   # chất lượng build index (cao → chính xác hơn)
FAISS_EF_SEARCH        = 128   # chất lượng search (cao → recall cao hơn)

# ── Split parameters ───────────────────────────────────────────────────────
TRAIN_RATIO = 0.8
VAL_RATIO   = 0.1
TEST_RATIO  = 0.1
SEED        = 42

# ── Output paths ───────────────────────────────────────────────────────────
OUTPUT_DIR = f"{DRIVE_BASE}/training_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)

print(f"Config ready. Output dir: {OUTPUT_DIR}")

FAISS version: 1.13.2
Config ready. Output dir: /content/drive/MyDrive/training_data


## 9.1 · Build FAISS index cho 5000 books

Sử dụng **FAISS HNSW** (Hierarchical Navigable Small World) — ANN index cho phép truy vấn cực nhanh (sub-millisecond cho 5K items) trong khi vẫn đạt recall gần 100%.

- Metric: **inner product** (tương đương cosine vì embeddings đã được normalize ở bước 2).
- Dùng lại `book_embs` đã cache để không phải re-encode.
- Pipeline hoàn toàn offline, không cần network/credentials như ChromaDB Cloud.

In [15]:
import time

# Documents (dùng cho export triplets cuối pipeline)
documents_all = [
    f"Title: {t} | Authors: {a} | Description: {d}"
    for t, a, d in zip(books_df["Title"], books_df["Authors"], books_df["Description"])
]
book_id_to_doc = {int(bid): doc for bid, doc in zip(books_df["book_id"], documents_all)}

# FAISS làm việc với row index (0..N-1), cần mapping row_idx ↔ book_id
faiss_idx_to_book_id = [int(bid) for bid in books_df["book_id"]]

# ── Build HNSW index ───────────────────────────────────────────────────────
# Embeddings đã normalize → inner product == cosine similarity.
dim = book_embs.shape[1]
book_embs_f32 = np.ascontiguousarray(book_embs, dtype=np.float32)

index = faiss.IndexHNSWFlat(dim, FAISS_HNSW_M, faiss.METRIC_INNER_PRODUCT)
index.hnsw.efConstruction = FAISS_EF_CONSTRUCTION
index.hnsw.efSearch       = FAISS_EF_SEARCH

t0 = time.time()
index.add(book_embs_f32)
build_time = time.time() - t0

print(f"FAISS HNSW index built: {index.ntotal} vectors, dim={dim}")
print(f"  M={FAISS_HNSW_M}, efConstruction={FAISS_EF_CONSTRUCTION}, efSearch={FAISS_EF_SEARCH}")
print(f"  Build time: {build_time:.2f}s")

FAISS HNSW index built: 5000 vectors, dim=1024
  M=32, efConstruction=200, efSearch=128
  Build time: 2.66s


## 9.2 · Retrieve Top-K candidates cho từng query

Dùng lại query embeddings đã cache, batch search trên FAISS index để lấy top-50 books gần nhất.
Với HNSW, mỗi batch xử lý gần như tức thời (vector hoá bằng SIMD của FAISS).

In [16]:
# Map selected queries → emb_idx (để reuse embedding đã cache)
mining_df = selected_df.merge(
    queries_df[["book_id", "query", "emb_idx"]].drop_duplicates(subset=["book_id", "query"]),
    on=["book_id", "query"],
    how="left",
)
mining_df = mining_df.dropna(subset=["emb_idx"]).reset_index(drop=True)
mining_df["emb_idx"] = mining_df["emb_idx"].astype(int)

print(f"Mining {len(mining_df)} queries across {mining_df['book_id'].nunique()} books.\n")

BATCH_QUERY = 512   # FAISS batch search rất rẻ → batch lớn để throughput tối đa
all_candidates = []

t0 = time.time()
for i in range(0, len(mining_df), BATCH_QUERY):
    batch = mining_df.iloc[i : i + BATCH_QUERY]
    q_embs = np.ascontiguousarray(
        query_embs[batch["emb_idx"].to_numpy()], dtype=np.float32
    )

    # FAISS trả similarity (inner product) + row index trong index
    sims, idxs = index.search(q_embs, TOP_K_RETRIEVE)

    for j, (_, row) in enumerate(batch.iterrows()):
        cand_ids = [faiss_idx_to_book_id[k] for k in idxs[j] if k != -1]
        all_candidates.append({
            "book_id":       int(row["book_id"]),
            "query":         row["query"],
            "candidate_ids": cand_ids,
            "distances":     [float(s) for s in sims[j][: len(cand_ids)]],
        })

    done = min(i + BATCH_QUERY, len(mining_df))
    if (i // BATCH_QUERY) % 5 == 0 or done == len(mining_df):
        print(f"  Retrieved {done}/{len(mining_df)}")

elapsed = time.time() - t0
print(
    f"\nDone. Got candidates for {len(all_candidates)} queries "
    f"in {elapsed:.2f}s ({len(all_candidates)/max(elapsed, 1e-9):.0f} q/s)."
)

Mining 14829 queries across 4943 books.

  Retrieved 512/14829
  Retrieved 3072/14829
  Retrieved 5632/14829
  Retrieved 8192/14829
  Retrieved 10752/14829
  Retrieved 13312/14829
  Retrieved 14829/14829

Done. Got candidates for 14829 queries in 9.15s (1621 q/s).


## 9.3 · Phân loại, bổ sung same-category & lọc false negatives

Với mỗi query:
- **Hard** (rank 1–10): negative khó, model dễ nhầm với positive
- **Medium** (rank 11–30): bổ sung để mô hình học dải rộng
- **Same-category**: sample random từ books cùng Category với positive — khó vì cùng chủ đề nhưng khác nội dung
- **False negative filter**: loại candidate có `cos_sim(neg, positive) > 0.85` (có thể là duplicate/near-duplicate)

Mỗi query cuối cùng có **7 negatives**: `4 hard + 2 same-cat + 1 medium` (fallback lấy từ rank 31+ nếu thiếu).

In [17]:
# Build lookup structures
book_id_to_idx      = {int(bid): i for i, bid in enumerate(books_df["book_id"])}
book_id_to_category = {int(bid): cat for bid, cat in zip(books_df["book_id"], books_df["Category"])}

cat_to_books: dict[str, list[int]] = {}
for bid, cat in zip(books_df["book_id"], books_df["Category"]):
    cat_to_books.setdefault(str(cat), []).append(int(bid))


def is_false_negative(neg_book_id: int, positive_emb: np.ndarray) -> bool:
    """Kiểm tra candidate có quá giống positive hay không (near-duplicate topic)."""
    neg_emb = book_embs[book_id_to_idx[neg_book_id]]
    return float(np.dot(neg_emb, positive_emb)) > FALSE_NEG_THRESHOLD


def pick_negatives(qc: dict, positive_book_id: int) -> tuple[list[int], dict]:
    """
    Select N_NEGATIVES negatives cho một query.

    Trả về (list negative book_ids, breakdown dict thống kê nguồn).
    """
    cand_ids     = qc["candidate_ids"]
    positive_emb = book_embs[book_id_to_idx[positive_book_id]]

    hard_neg, medium_neg = [], []
    for rank, bid in enumerate(cand_ids, start=1):
        if bid == positive_book_id:
            continue
        if is_false_negative(bid, positive_emb):
            continue
        if rank <= HARD_RANK_MAX and len(hard_neg) < N_HARD:
            hard_neg.append(bid)
        elif HARD_RANK_MAX < rank <= MEDIUM_RANK_MAX and len(medium_neg) < N_MEDIUM:
            medium_neg.append(bid)

    # Same-category negatives (khác positive, khác các neg đã chọn)
    chosen = set(hard_neg) | set(medium_neg) | {positive_book_id}
    pos_cat = book_id_to_category.get(positive_book_id, "")
    same_cat_pool = [
        b for b in cat_to_books.get(str(pos_cat), [])
        if b not in chosen and not is_false_negative(b, positive_emb)
    ]
    same_cat_neg = random.sample(same_cat_pool, min(N_SAME_CAT, len(same_cat_pool)))

    negatives = list(hard_neg) + list(same_cat_neg) + list(medium_neg)

    # Fallback: pad với easy (rank 31+) nếu vẫn thiếu
    if len(negatives) < N_NEGATIVES:
        existing = set(negatives) | {positive_book_id}
        for bid in cand_ids[MEDIUM_RANK_MAX:]:
            if bid in existing:
                continue
            if is_false_negative(bid, positive_emb):
                continue
            negatives.append(bid)
            existing.add(bid)
            if len(negatives) >= N_NEGATIVES:
                break

    breakdown = {
        "n_hard":     len(hard_neg),
        "n_same_cat": len(same_cat_neg),
        "n_medium":   len(medium_neg),
        "n_padded":   max(0, len(negatives) - N_HARD - N_SAME_CAT - N_MEDIUM),
    }
    return negatives[:N_NEGATIVES], breakdown


# Run mining
triplets = []
stats = {
    "positive_in_top1"    : 0,
    "positive_in_top10"   : 0,
    "positive_in_top50"   : 0,
    "positive_not_in_top50": 0,
    "insufficient_neg"    : 0,
    "total_hard"          : 0,
    "total_same_cat"      : 0,
    "total_medium"        : 0,
    "total_padded"        : 0,
}

for qc in all_candidates:
    pos_id = qc["book_id"]
    cand_ids = qc["candidate_ids"]

    if pos_id in cand_ids:
        rank = cand_ids.index(pos_id) + 1
        stats["positive_in_top50"] += 1
        if rank == 1:
            stats["positive_in_top1"] += 1
        if rank <= 10:
            stats["positive_in_top10"] += 1
    else:
        stats["positive_not_in_top50"] += 1

    negs, breakdown = pick_negatives(qc, pos_id)

    if len(negs) < N_NEGATIVES:
        stats["insufficient_neg"] += 1
        continue

    stats["total_hard"]     += breakdown["n_hard"]
    stats["total_same_cat"] += breakdown["n_same_cat"]
    stats["total_medium"]   += breakdown["n_medium"]
    stats["total_padded"]   += breakdown["n_padded"]

    triplets.append({
        "book_id": pos_id,
        "query":   qc["query"],
        "pos":     [book_id_to_doc[pos_id]],
        "neg":     [book_id_to_doc[b] for b in negs],
    })

n = len(all_candidates) or 1
print("=== Hard Negative Mining Stats ===")
print(f"  Total queries            : {len(all_candidates)}")
print(f"  Positive in top-1        : {stats['positive_in_top1']:>6} ({stats['positive_in_top1']/n*100:5.1f}%)")
print(f"  Positive in top-10       : {stats['positive_in_top10']:>6} ({stats['positive_in_top10']/n*100:5.1f}%)")
print(f"  Positive in top-50       : {stats['positive_in_top50']:>6} ({stats['positive_in_top50']/n*100:5.1f}%)")
print(f"  Positive NOT in top-50   : {stats['positive_not_in_top50']:>6} ({stats['positive_not_in_top50']/n*100:5.1f}%)")
print(f"  Skipped (insufficient neg): {stats['insufficient_neg']}")
print(f"  Valid triplets           : {len(triplets)}")
print()
m = len(triplets) or 1
print("Negative source breakdown (avg per query):")
print(f"  Hard       : {stats['total_hard']/m:.2f}")
print(f"  Same-cat   : {stats['total_same_cat']/m:.2f}")
print(f"  Medium     : {stats['total_medium']/m:.2f}")
print(f"  Fallback   : {stats['total_padded']/m:.2f}")

=== Hard Negative Mining Stats ===
  Total queries            : 14829
  Positive in top-1        :   8668 ( 58.5%)
  Positive in top-10       :  12550 ( 84.6%)
  Positive in top-50       :  13882 ( 93.6%)
  Positive NOT in top-50   :    947 (  6.4%)
  Skipped (insufficient neg): 0
  Valid triplets           : 14829

Negative source breakdown (avg per query):
  Hard       : 4.00
  Same-cat   : 1.66
  Medium     : 1.00
  Fallback   : 0.00


## 9.4 · Sanity check một triplet

Đọc 1 sample bằng mắt để xác nhận negatives có reasonable (khác topic với positive nhưng vẫn là sách thật).

In [18]:
sample = random.choice(triplets)
print(f"QUERY   : {sample['query']}")
print(f"POSITIVE: {sample['pos'][0][:200]}...")
print()
print("NEGATIVES:")
for i, neg in enumerate(sample["neg"], 1):
    print(f"  {i}. {neg[:180]}...")

QUERY   : sách nào giúp mẹ cân bằng công việc và sức khỏe
POSITIVE: Title: The Girlfriends' Guide to Surviving the First Year of Motherhood, Packaging May Vary | Authors: By Iovine, Vicki | Description: When it comes to your new baby, everyone from Dr. Spock to Dr. Br...

NEGATIVES:
  1. Title: Life Matters : Creating a Dynamic Balance of Work, Family, Time & Money | Authors: By Merrill, A. Roger and Merrill, Rebecca R. | Description: Praise for Life Matters:  "A g...
  2. Title: What Every Mom Needs | Authors: By Morgan, Elisa and Kuykendall, Carol | Description: 'When will I ever get time for myself? Do I ever get to be someone besides &apos;Mom&ap...
  3. Title: Mother to Daughter | Authors: By Harrison, Harry H., Jr. and Harrison, Melissa | Description: Warm and fuzzy, anchored in values, and filled with simple words of wisdom, thi...
  4. Title: The Don't Sweat Guide for Parents: Reduce Stress and Enjoy Your Kids More (Don't Sweat Guides) | Authors: By Carlson, Richard | Descripti

## 9.5 · Export BGE format + split train/val/test

- Split theo **`book_id`** (không theo query) để tránh data leakage.
- Stratify theo **Category** để phân phối đều giữa các split.
- Output format chuẩn BGE: mỗi dòng JSONL là `{"query": ..., "pos": [...], "neg": [...]}`.

In [20]:
# ── Book-level split (stratified by Category) ──────────────────────────────
book_ids_used = sorted({t["book_id"] for t in triplets})
book_meta = books_df[books_df["book_id"].isin(book_ids_used)][["book_id", "Category"]].copy()


def build_strat_key(df: pd.DataFrame, col: str, min_count: int) -> pd.Series:
    """Gộp các class có <min_count members thành '_OTHER_' để stratify không lỗi.

    StratifiedShuffleSplit yêu cầu mỗi class có ≥2 samples. Ở split đầu tiên
    (80/20) cần min_count=5 (để 20% còn ≥1 cho cả val và test ở split sau).
    Ở split thứ hai (50/50 của 20%) cần min_count=2.
    """
    counts = df[col].value_counts()
    rare = set(counts[counts < min_count].index)
    return df[col].apply(lambda c: "_OTHER_" if c in rare else c)


# Split 1: train vs temp(val+test). Class cần ≥5 để sau khi lấy 20% còn ≥1
# cho mỗi bên val/test ở split kế tiếp; thêm buffer tránh edge-case làm tròn.
book_meta["strat_key"] = build_strat_key(book_meta, "Category", min_count=5)

train_books, temp_books = train_test_split(
    book_meta,
    test_size=VAL_RATIO + TEST_RATIO,
    stratify=book_meta["strat_key"],
    random_state=SEED,
)

# Split 2: re-compute strat_key trên temp_books — một số class sau split 1
# có thể chỉ còn 1 member, phải merge vào "_OTHER_" để tránh ValueError.
temp_books = temp_books.copy()
temp_books["strat_key"] = build_strat_key(temp_books, "Category", min_count=2)

val_books, test_books = train_test_split(
    temp_books,
    test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO),
    stratify=temp_books["strat_key"],
    random_state=SEED,
)

train_ids = set(train_books["book_id"].astype(int))
val_ids   = set(val_books["book_id"].astype(int))
test_ids  = set(test_books["book_id"].astype(int))


def save_jsonl(records: list[dict], path: str) -> None:
    """Persist BGE triplets: {query, pos, neg} — drop internal book_id field."""
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            out = {"query": r["query"], "pos": r["pos"], "neg": r["neg"]}
            f.write(json.dumps(out, ensure_ascii=False) + "\n")


train_triplets = [t for t in triplets if t["book_id"] in train_ids]
val_triplets   = [t for t in triplets if t["book_id"] in val_ids]
test_triplets  = [t for t in triplets if t["book_id"] in test_ids]

train_path = f"{OUTPUT_DIR}/train.jsonl"
val_path   = f"{OUTPUT_DIR}/val.jsonl"
test_path  = f"{OUTPUT_DIR}/test.jsonl"

save_jsonl(train_triplets, train_path)
save_jsonl(val_triplets,   val_path)
save_jsonl(test_triplets,  test_path)

print("━" * 60)
print("TRAINING DATASET BUILD COMPLETE")
print("━" * 60)
print(f"  Train : {len(train_books):>5} books  |  {len(train_triplets):>6} queries  →  train.jsonl")
print(f"  Val   : {len(val_books):>5} books  |  {len(val_triplets):>6} queries  →  val.jsonl")
print(f"  Test  : {len(test_books):>5} books  |  {len(test_triplets):>6} queries  →  test.jsonl")
print(f"  Neg/Q : {N_NEGATIVES}")
print(f"  Output: {OUTPUT_DIR}")
print("━" * 60)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
TRAINING DATASET BUILD COMPLETE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Train :  3954 books  |   11862 queries  →  train.jsonl
  Val   :   494 books  |    1482 queries  →  val.jsonl
  Test  :   495 books  |    1485 queries  →  test.jsonl
  Neg/Q : 7
  Output: /content/drive/MyDrive/training_data
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


## 9.6 · Leakage & distribution check

Kiểm tra nhanh: không có `book_id` xuất hiện ở 2 splits, phân phối Category giữa các split tương đồng.

In [21]:
assert not (train_ids & val_ids),  "Leakage: train ∩ val"
assert not (train_ids & test_ids), "Leakage: train ∩ test"
assert not (val_ids & test_ids),   "Leakage: val ∩ test"
print("✓ No book_id leakage across splits.\n")

# Top categories per split
def top_cat_dist(df, k=5):
    d = df["Category"].value_counts(normalize=True).head(k)
    return {c: round(p * 100, 1) for c, p in d.items()}

print("Top-5 Category distribution (%):")
print(f"  Train: {top_cat_dist(train_books)}")
print(f"  Val  : {top_cat_dist(val_books)}")
print(f"  Test : {top_cat_dist(test_books)}")

# Overall quality
total_queries = len(train_triplets) + len(val_triplets) + len(test_triplets)
total_books   = len(train_ids | val_ids | test_ids)
print(f"\n  Books used : {total_books}")
print(f"  Queries    : {total_queries}")
print(f"  Avg Q/book : {total_queries / max(total_books, 1):.2f}")
print(f"  Total neg  : {total_queries * N_NEGATIVES:,}")

✓ No book_id leakage across splits.

Top-5 Category distribution (%):
  Train: {' Fiction , General': 7.1, ' Fiction , Mystery & Detective , General': 3.9, ' Fiction , Literary': 3.1, ' Biography & Autobiography , General': 1.7, ' Political Science , General': 1.4}
  Val  : {' Fiction , General': 7.1, ' Fiction , Mystery & Detective , General': 3.8, ' Fiction , Literary': 3.0, ' Biography & Autobiography , General': 1.6, ' Political Science , General': 1.4}
  Test : {' Fiction , General': 7.1, ' Fiction , Mystery & Detective , General': 3.8, ' Fiction , Literary': 3.2, ' Biography & Autobiography , General': 1.8, ' Political Science , General': 1.4}

  Books used : 4943
  Queries    : 14829
  Avg Q/book : 3.00
  Total neg  : 103,803
